# PhysIQ – Task 2: Stability Judgment

Given a 2D arrangement of objects, determine whether the configuration will remain stable or collapse, identify the first object to fail, and describe the failure direction.

**Format:**
```
STABLE: yes/no
FIRST_FAILURE: <object_id> (if unstable)
FAILURE_DIRECTION: left/right/topples (if unstable)
```
**Scoring:** Judgment (0.5) + failure ID (0.3) + failure description (0.2)

In [ ]:
# ── Cell 1: Setup & imports ──────────────────────────────────────────────────
import sys, os, json, re
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'Kaggle_agi_bench'))

import kaggle_benchmarks as kbench
print('kaggle-benchmarks', kbench.__version__)

In [ ]:
# ── Cell 2: Physics engine & scenario library ────────────────────────────────
from physiq.engine import PhysIQWorld
from physiq.formats import build_prompt
from physiq.generation import compute_ground_truth, validate_scenario, _task_seed
from physiq.scoring import score_stability
from physiq.templates.stability import STABILITY_TEMPLATES
from physiq.templates import SCENARIO_COUNTS

print(f'Stability templates: {len(STABILITY_TEMPLATES)}')

In [ ]:
# ── Cell 3: Generate stability scenarios ────────────────────────────────────
MASTER_SEED = 42
task_type = 'stability'
counts = SCENARIO_COUNTS[task_type]

task_seed = _task_seed(task_type, MASTER_SEED)
rng = np.random.RandomState(task_seed)

scenarios = []
for difficulty in ['easy', 'medium', 'hard']:
    target = counts[difficulty]
    collected = []
    attempt = 0
    while len(collected) < target and attempt < target * 3:
        seed = int(rng.randint(0, 2**31))
        template = STABILITY_TEMPLATES[attempt % len(STABILITY_TEMPLATES)]
        attempt += 1
        try:
            s = template.generate(difficulty, seed)
        except Exception:
            continue
        if validate_scenario(s, task_type):
            s['_ground_truth'] = compute_ground_truth(s, task_type)
            collected.append(s)
    scenarios.extend(collected)
    print(f'  {difficulty}: {len(collected)}/{target}')

print(f'Total stability scenarios: {len(scenarios)}')

In [ ]:
# ── Cell 4: Build evaluation DataFrame ──────────────────────────────────────
rows = []
for s in scenarios:
    gt = s['_ground_truth']
    for fmt in ['json', 'ascii', 'nl']:
        rows.append({
            'scenario_id':    s['id'],
            'difficulty':     s['difficulty'],
            'representation': fmt,
            'prompt':         build_prompt(s, fmt, task_type),
            'ground_truth':   json.dumps(gt),
        })

df_stability = pd.DataFrame(rows)
print(df_stability.shape)
df_stability.head(2)

In [ ]:
# ── Cell 5: Scoring helpers ──────────────────────────────────────────────────
_FAILURE_KEYWORDS = {
    'left', 'right', 'topple', 'topples', 'fall', 'falls',
    'slide', 'slides', 'collapse', 'collapses', 'tip', 'tips',
    'rotate', 'rotates', 'overturn', 'overturns', 'lean', 'leans',
}


def _parse_stability_response(response: str) -> dict:
    """Parse model stability response into structured fields."""
    resp = response.lower()

    # STABLE: yes/no
    stable_match = re.search(r'stable\s*:\s*(yes|no|true|false|stable|unstable)', resp)
    if stable_match:
        val = stable_match.group(1)
        is_stable = val in ('yes', 'true', 'stable')
    else:
        # Heuristic: look for explicit instability claim
        is_stable = 'unstable' not in resp and 'collapse' not in resp and 'fall' not in resp

    # FIRST_FAILURE: object_id
    ff_match = re.search(r'first_failure\s*:\s*([\w_]+)', resp)
    first_failure = ff_match.group(1) if ff_match else None

    # FAILURE_DIRECTION: left/right/topples
    fd_match = re.search(r'failure_direction\s*:\s*(\w+)', resp)
    failure_dir = fd_match.group(1) if fd_match else None

    # Keyword fallback for direction
    if failure_dir is None:
        for kw in ['left', 'right', 'topples', 'topple']:
            if kw in resp:
                failure_dir = kw
                break

    return {
        'stable': is_stable,
        'first_failure': first_failure,
        'failure_direction': failure_dir,
        'raw': response,
    }


def _score_stability_response(response: str, ground_truth_json: str) -> float:
    gt = json.loads(ground_truth_json)
    parsed = _parse_stability_response(response)

    score = 0.0

    # 1) Stability judgment (0.5)
    gt_stable = gt.get('stable', True)
    if parsed['stable'] == gt_stable:
        score += 0.5

    if gt_stable:
        # Correct stable prediction gets full 0.5, nothing more to score
        return min(score, 1.0)

    # 2) Failure identification (0.3) - only if actually unstable
    gt_failures = gt.get('failure_events', [])
    if gt_failures and parsed['first_failure'] is not None:
        gt_obj = gt_failures[0].get('object_id', '')
        # 0.15 for object ID match
        if parsed['first_failure'].lower() in gt_obj.lower() or gt_obj.lower() in parsed['first_failure'].lower():
            score += 0.15
        # 0.15 for direction match
        gt_dir = gt_failures[0].get('direction', '')
        if parsed['failure_direction'] and gt_dir and parsed['failure_direction'] in gt_dir:
            score += 0.15

    # 3) Final state description (0.2) - keyword matching
    resp_lower = response.lower()
    kw_hits = sum(1 for kw in _FAILURE_KEYWORDS if kw in resp_lower)
    score += min(0.2, kw_hits * 0.05)

    return min(score, 1.0)


# Sanity check
gt_sample = df_stability.iloc[0]['ground_truth']
gt_dict = json.loads(gt_sample)
if gt_dict['stable']:
    fake = 'STABLE: yes'
else:
    obj = gt_dict['failure_events'][0]['object_id'] if gt_dict['failure_events'] else 'block'
    d = gt_dict['failure_events'][0]['direction'] if gt_dict['failure_events'] else 'left'
    fake = f'STABLE: no\nFIRST_FAILURE: {obj}\nFAILURE_DIRECTION: {d}\nThe block topples left'
print('Perfect answer score:', _score_stability_response(fake, gt_sample))

In [ ]:
# ── Cell 6: Task definition ──────────────────────────────────────────────────
@kbench.task(
    name='PhysIQ Stability Judgment',
    description=(
        'Given a 2D arrangement of physics objects, determine whether it '
        'will remain stable or collapse. If unstable, identify the first '
        'object to fail and the direction. '
        'Tests spatial reasoning about balance and support.'
    ),
)
def physiq_stability(llm: kbench.LLMChat, prompt: str, ground_truth: str) -> float:
    """Predict stability of a 2D physics arrangement."""
    response = llm.prompt(prompt)
    return _score_stability_response(response, ground_truth)

In [ ]:
# ── Cell 7: Dry-run sanity check ─────────────────────────────────────────────
required_cols = {'prompt', 'ground_truth'}
assert required_cols.issubset(df_stability.columns)
print('DataFrame columns OK:', list(df_stability.columns))
print('Rows:', len(df_stability))
stable_frac = sum(
    json.loads(r)['stable'] for r in df_stability['ground_truth'].unique()
) / len(df_stability['ground_truth'].unique())
print(f'Stable fraction: {stable_frac:.2f} (target ~0.50)')

In [ ]:
# ── Cell 8: Evaluation run ───────────────────────────────────────────────────
try:
    llm = kbench.llm
except AttributeError:
    print('No Kaggle LLM available (local run). Skipping evaluation.')
    llm = None

if llm is not None:
    results = []
    for _, row in df_stability.iterrows():
        run = physiq_stability.run(
            llm=llm,
            prompt=row['prompt'],
            ground_truth=row['ground_truth'],
        )
        results.append({
            'scenario_id':    row['scenario_id'],
            'difficulty':     row['difficulty'],
            'representation': row['representation'],
            'score':          run.result,
        })
    df_results = pd.DataFrame(results)
    os.makedirs('../outputs', exist_ok=True)
    df_results.to_csv('../outputs/task2_stability_results.csv', index=False)
    print('Mean score:', df_results['score'].mean())
    print(df_results.groupby(['difficulty', 'representation'])['score'].mean().unstack())

In [ ]:
# ── Cell 9: Results analysis ─────────────────────────────────────────────────
if llm is not None and 'df_results' in dir():
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df_results.groupby('difficulty')['score'].mean().plot(
        kind='bar', ax=axes[0], color=['#4CAF50', '#FF9800', '#F44336'])
    axes[0].set_title('Stability Score by Difficulty')
    axes[0].set_ylabel('Mean Score (0-1)')
    axes[0].set_ylim(0, 1)

    df_results.groupby('representation')['score'].mean().plot(
        kind='bar', ax=axes[1], color=['#2196F3', '#9C27B0', '#00BCD4'])
    axes[1].set_title('Stability Score by Format')
    axes[1].set_ylabel('Mean Score (0-1)')
    axes[1].set_ylim(0, 1)

    plt.tight_layout()
    plt.savefig('../outputs/task2_stability_results.png', dpi=150)
    plt.show()

In [ ]:
# ── Cell 10: Choose this task for the leaderboard ────────────────────────────
%choose physiq_stability